In [1]:
%pip install pymongo
"""
Step 1 – Data Preparation
Import Reviews_withURL.csv into MongoDB collection 'reviews'
inside database 'amazon_reviews'.
"""

import pandas as pd
from pymongo import MongoClient, UpdateOne
import time

CSV_PATH = "./Reviews_withURL.csv"
MONGO_URI = "mongodb://localhost:27017"
DB_NAME   = "amazon_reviews"
COL_NAME  = "reviews"

print("Loading CSV …")
df = pd.read_csv(CSV_PATH, index_col=0)          # first col is unnamed row-index
print(f"  Rows: {len(df):,}  |  Columns: {list(df.columns)}")

# ── Clean up ──────────────────────────────────────────────────────────────────
# Convert timestamp (Unix seconds) to datetime
df["ReviewDate"] = pd.to_datetime(df["Time"], unit="s")

# Derive a new field: SentimentLabel  (Score 4-5 → Positive, 3 → Neutral, 1-2 → Negative)
def label(score):
    if score >= 4:  return "Positive"
    if score == 3:  return "Neutral"
    return "Negative"

df["SentimentLabel"] = df["Score"].apply(label)

# Helpfulness ratio (avoid division by zero)
df["HelpfulnessRatio"] = df.apply(
    lambda r: round(r["HelpfulnessNumerator"] / r["HelpfulnessDenominator"], 4)
    if r["HelpfulnessDenominator"] > 0 else None,
    axis=1
)

records = df.to_dict(orient="records")

# ── Import ────────────────────────────────────────────────────────────────────
client = MongoClient(MONGO_URI)
db     = client[DB_NAME]
col    = db[COL_NAME]

col.drop()          # fresh import each run
print("Inserting documents …")
t0 = time.time()
BATCH = 10_000
for i in range(0, len(records), BATCH):
    col.insert_many(records[i:i+BATCH])
    if (i // BATCH) % 10 == 0:
        print(f"  {i+BATCH:,} / {len(records):,} inserted …")

elapsed = time.time() - t0
print(f"\nImport complete in {elapsed:.1f}s")
print(f"  Total documents: {col.count_documents({}):,}")

# ── Indexes ───────────────────────────────────────────────────────────────────
col.create_index("ProductId")
col.create_index("UserId")
col.create_index("Score")
col.create_index("SentimentLabel")
col.create_index("ReviewDate")
print("Indexes created.")
client.close()


Note: you may need to restart the kernel to use updated packages.
Loading CSV …
  Rows: 568,454  |  Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text', 'ProductURL']
Inserting documents …
  10,000 / 568,454 inserted …
  110,000 / 568,454 inserted …
  210,000 / 568,454 inserted …
  310,000 / 568,454 inserted …
  410,000 / 568,454 inserted …
  510,000 / 568,454 inserted …

Import complete in 5.1s
  Total documents: 568,454
Indexes created.


In [5]:
"""
Step 2 – Natural Language Processing
1.  Efficient word-frequency counting with Counter (no NLTK tokeniser overhead)
2.  TF-IDF keyword extraction per sentiment class
3.  Named-entity style keyword extraction using noun-phrase chunking (NLTK)
4.  Bigram analysis
"""

import re, string
from collections import Counter
import pandas as pd
from pymongo import MongoClient
import nltk

# Download once
for pkg in ["punkt", "stopwords", "averaged_perceptron_tagger", "punkt_tab",
            "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk import word_tokenize, pos_tag, ne_chunk, ngrams
from sklearn.feature_extraction.text import TfidfVectorizer
import json, os

MONGO_URI = "mongodb://localhost:27017"
DB_NAME   = "amazon_reviews"
COL_NAME  = "reviews"
OUT_DIR   = "./nlp_results"
os.makedirs(OUT_DIR, exist_ok=True)

# ─── helpers ──────────────────────────────────────────────────────────────────
STOP = set(stopwords.words("english"))
STOP |= {"br", "amazon", "product", "one", "would", "get", "also", "like", "much",
          "really", "even", "made", "make", "use", "used", "great", "good", "food",
          "item", "buy", "bought", "still", "taste", "flavor", "well", "got", "back"}

def clean_tokens(text: str) -> list:
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)          # strip HTML tags
    text = re.sub(r"[^a-z\s]", " ", text)       # keep only letters
    tokens = text.split()
    return [t for t in tokens if t not in STOP and len(t) > 2]

# ─── load data from MongoDB ───────────────────────────────────────────────────
print("Connecting to MongoDB …")
client = MongoClient(MONGO_URI)
col    = client[DB_NAME][COL_NAME]

print("Fetching reviews …")
cursor = col.find({}, {"Text": 1, "Summary": 1, "Score": 1, "SentimentLabel": 1, "_id": 0})
rows   = list(cursor)
client.close()
print(f"  Loaded {len(rows):,} documents")

df = pd.DataFrame(rows)
df["combined"] = df["Summary"].fillna("") + " " + df["Text"].fillna("")

# ════════════════════════════════════════════════════════════════════════════════
# TASK A – Overall word frequency (most efficient: single Counter pass)
# ════════════════════════════════════════════════════════════════════════════════
print("\n[A] Counting word frequencies …")
total_counter: Counter = Counter()
for text in df["combined"]:
    total_counter.update(clean_tokens(text))

top100 = total_counter.most_common(100)
wf_df  = pd.DataFrame(top100, columns=["word", "count"])
wf_df.to_csv(f"{OUT_DIR}/word_freq_top100.csv", index=False)
print(f"  Top 10: {top100[:10]}")

# ════════════════════════════════════════════════════════════════════════════════
# TASK B – Word frequency per sentiment (efficient: grouped Counter)
# ════════════════════════════════════════════════════════════════════════════════
print("\n[B] Word freq by sentiment …")
sent_counters = {}
for label, group in df.groupby("SentimentLabel"):
    c = Counter()
    for text in group["combined"]:
        c.update(clean_tokens(text))
    sent_counters[label] = c
    top5 = c.most_common(5)
    print(f"  {label}: {top5}")
    pd.DataFrame(c.most_common(50), columns=["word","count"]).to_csv(
        f"{OUT_DIR}/word_freq_{label.lower()}.csv", index=False)

# ════════════════════════════════════════════════════════════════════════════════
# TASK C – TF-IDF keyword extraction per sentiment class
# ════════════════════════════════════════════════════════════════════════════════
print("\n[C] TF-IDF per sentiment …")
sentiment_docs = {}
for label, group in df.groupby("SentimentLabel"):
    # Concatenate all reviews in that class into one large document
    sentiment_docs[label] = " ".join(group["combined"].fillna("").tolist())

tfidf = TfidfVectorizer(max_features=200, stop_words="english",
                        ngram_range=(1, 2), min_df=1)
tfidf_matrix = tfidf.fit_transform(list(sentiment_docs.values()))
feature_names = tfidf.get_feature_names_out()
labels_list   = list(sentiment_docs.keys())

tfidf_results = {}
for i, label in enumerate(labels_list):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:20]
    top_keywords = [(feature_names[j], round(float(scores[j]), 4)) for j in top_idx]
    tfidf_results[label] = top_keywords
    print(f"  {label} top keywords: {top_keywords[:5]}")

with open(f"{OUT_DIR}/tfidf_keywords.json", "w") as f:
    json.dump(tfidf_results, f, indent=2)

# ════════════════════════════════════════════════════════════════════════════════
# TASK D – Bigram analysis (top 50 bigrams overall)
# ════════════════════════════════════════════════════════════════════════════════
print("\n[D] Bigram analysis …")
bigram_counter: Counter = Counter()
SAMPLE_N = 50_000   # use a sample for speed
sample = df["combined"].sample(n=SAMPLE_N, random_state=42)
for text in sample:
    tokens = clean_tokens(text)
    bigram_counter.update(ngrams(tokens, 2))

top_bigrams = [(f"{a} {b}", c) for (a, b), c in bigram_counter.most_common(50)]
pd.DataFrame(top_bigrams, columns=["bigram", "count"]).to_csv(
    f"{OUT_DIR}/bigrams_top50.csv", index=False)
print(f"  Top 10 bigrams: {top_bigrams[:10]}")

# ════════════════════════════════════════════════════════════════════════════════
# TASK E – Demonstrate searching for a specific word in MongoDB (query approach)
# ════════════════════════════════════════════════════════════════════════════════
SEARCH_WORD = "delicious"
print(f"\n[E] Counting appearance of '{SEARCH_WORD}' via regex query …")
client2 = MongoClient(MONGO_URI)
col2    = client2[DB_NAME][COL_NAME]

regex_count = col2.count_documents(
    {"Text": {"$regex": SEARCH_WORD, "$options": "i"}}
)
print(f"  '{SEARCH_WORD}' appears in {regex_count:,} reviews (MongoDB regex query)")

# Also demonstrate text-index approach (most efficient for full-text search)
try:
    col2.create_index([("Text", "text"), ("Summary", "text")])
    text_result = col2.find(
        {"$text": {"$search": SEARCH_WORD}},
        {"score": {"$meta": "textScore"}, "Summary": 1, "Score": 1, "_id": 0}
    ).sort([("score", {"$meta": "textScore"})]).limit(3)
    print("  Text-index top-3 results:")
    for doc in text_result:
        print(f"    Score={doc['Score']} | {doc['Summary'][:60]}")
except Exception as e:
    print(f"  Text index note: {e}")

client2.close()

print("\nStep 2 NLP complete. Results saved to", OUT_DIR)


Connecting to MongoDB …
Fetching reviews …
  Loaded 568,454 documents

[A] Counting word frequencies …
  Top 10: [('coffee', 191977), ('tea', 160364), ('love', 155159), ('best', 110362), ('little', 87888), ('time', 87136), ('price', 87099), ('dog', 83550), ('better', 78176), ('tried', 77687)]

[B] Word freq by sentiment …
  Negative: [('coffee', 24822), ('tea', 17824), ('dog', 13582), ('bad', 12819), ('first', 12225)]
  Neutral: [('coffee', 19729), ('tea', 11606), ('better', 9208), ('little', 8897), ('water', 7709)]
  Positive: [('coffee', 147426), ('love', 138819), ('tea', 130934), ('best', 101363), ('price', 70353)]

[C] TF-IDF per sentiment …
  Negative top keywords: [('br', 0.6168), ('like', 0.2834), ('br br', 0.2472), ('product', 0.2213), ('taste', 0.2131)]
  Neutral top keywords: [('br', 0.6314), ('br br', 0.2645), ('like', 0.2631), ('good', 0.2292), ('taste', 0.2061)]
  Positive top keywords: [('br', 0.5584), ('great', 0.2574), ('good', 0.2383), ('like', 0.2274), ('br br', 0.222

In [6]:
"""
Step 3 – Analysis & Visualization
Produces 8 publication-quality charts saved to /project/figures/
"""

import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pymongo import MongoClient
from collections import Counter
from wordcloud import WordCloud
import re

warnings.filterwarnings("ignore")
FIG_DIR = "./figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ── Palette ───────────────────────────────────────────────────────────────────
PALETTE = {"Positive": "#2ecc71", "Neutral": "#f39c12", "Negative": "#e74c3c"}
BLUE    = "#3498db"
DARK    = "#2c3e50"
sns.set_theme(style="whitegrid", font_scale=1.15)

# ── Load data ─────────────────────────────────────────────────────────────────
print("Loading data from MongoDB …")
client = MongoClient("mongodb://localhost:27017")
col    = client["amazon_reviews"]["reviews"]
cursor = col.find({}, {"Score": 1, "SentimentLabel": 1, "ReviewDate": 1,
                        "HelpfulnessNumerator": 1, "HelpfulnessDenominator": 1,
                        "HelpfulnessRatio": 1, "combined": 1, "Text": 1,
                        "Summary": 1, "_id": 0})
rows = list(cursor)
client.close()

df = pd.DataFrame(rows)
df["ReviewDate"] = pd.to_datetime(df["ReviewDate"], errors="coerce")
df["Year"]  = df["ReviewDate"].dt.year
df["Month"] = df["ReviewDate"].dt.to_period("M").dt.to_timestamp()
df["combined"] = df["Summary"].fillna("") + " " + df["Text"].fillna("")

print(f"  {len(df):,} rows loaded")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 1 – Score Distribution
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 5))
score_counts = df["Score"].value_counts().sort_index()
bars = ax.bar(score_counts.index, score_counts.values,
              color=[PALETTE["Negative"], PALETTE["Negative"],
                     PALETTE["Neutral"],  PALETTE["Positive"], PALETTE["Positive"]],
              edgecolor="white", width=0.6)
for bar, val in zip(bars, score_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f"{val:,}", ha="center", va="bottom", fontsize=10, color=DARK)
ax.set_xlabel("Star Rating", fontsize=12)
ax.set_ylabel("Number of Reviews", fontsize=12)
ax.set_title("Distribution of Star Ratings", fontsize=14, fontweight="bold", color=DARK)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig1_score_distribution.png", dpi=150)
plt.close()
print("  Fig 1 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 2 – Sentiment Pie Chart
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 5))
sent_counts = df["SentimentLabel"].value_counts()
colors = [PALETTE[l] for l in sent_counts.index]
wedges, texts, autotexts = ax.pie(
    sent_counts.values, labels=sent_counts.index,
    colors=colors, autopct="%1.1f%%",
    startangle=140, pctdistance=0.75,
    wedgeprops={"edgecolor": "white", "linewidth": 2})
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold")
ax.set_title("Sentiment Distribution", fontsize=14, fontweight="bold", color=DARK)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig2_sentiment_pie.png", dpi=150)
plt.close()
print("  Fig 2 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 3 – Review Volume Over Time (yearly)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
yearly = df.groupby("Year").size().reset_index(name="count")
yearly = yearly[yearly["Year"].between(2000, 2012)]
ax.fill_between(yearly["Year"], yearly["count"], alpha=0.25, color=BLUE)
ax.plot(yearly["Year"], yearly["count"], marker="o", color=BLUE, linewidth=2.5)
for _, row in yearly.iterrows():
    ax.text(row["Year"], row["count"] + 1500, f"{int(row['count']):,}",
            ha="center", fontsize=8.5, color=DARK)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Number of Reviews", fontsize=12)
ax.set_title("Review Volume Growth (2000 – 2012)", fontsize=14, fontweight="bold", color=DARK)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xticks(range(2000, 2013))
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig3_volume_over_time.png", dpi=150)
plt.close()
print("  Fig 3 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 4 – Average Score Over Time
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
avg_score = df[df["Year"].between(2000, 2012)].groupby("Year")["Score"].mean()
ax.plot(avg_score.index, avg_score.values, marker="s", color="#9b59b6",
        linewidth=2.5, markersize=7)
ax.axhline(df["Score"].mean(), linestyle="--", color="grey", alpha=0.6,
           label=f"Overall mean: {df['Score'].mean():.2f}")
ax.set_ylim(1, 5.2)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Average Star Rating", fontsize=12)
ax.set_title("Average Rating per Year", fontsize=14, fontweight="bold", color=DARK)
ax.set_xticks(range(2000, 2013))
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig4_avg_score_over_time.png", dpi=150)
plt.close()
print("  Fig 4 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 5 – Top 20 Words (bar chart)
# ══════════════════════════════════════════════════════════════════════════════
wf_path = "./nlp_results/word_freq_top100.csv"
if os.path.exists(wf_path):
    wf_df = pd.read_csv(wf_path).head(20)
else:
    # quick fallback
    from collections import Counter
    stop = {"the","and","is","in","it","of","a","to","i","this","have","for",
            "with","was","my","but","are","be","that","not","they","we","so",
            "br","amazon","product","food","good","great"}
    c = Counter()
    for text in df["combined"].sample(50000, random_state=1):
        c.update([w for w in re.sub(r"[^a-z ]", " ", text.lower()).split()
                  if w not in stop and len(w) > 2])
    wf_df = pd.DataFrame(c.most_common(20), columns=["word","count"])

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=wf_df, y="word", x="count", palette="Blues_r", ax=ax)
ax.set_xlabel("Frequency", fontsize=12)
ax.set_ylabel("Word", fontsize=12)
ax.set_title("Top 20 Most Frequent Words in Reviews", fontsize=14,
             fontweight="bold", color=DARK)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig5_top20_words.png", dpi=150)
plt.close()
print("  Fig 5 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 6 – Word Cloud (positive reviews)
# ══════════════════════════════════════════════════════════════════════════════
pos_path = "./nlp_results/word_freq_positive.csv"
if os.path.exists(pos_path):
    pos_df = pd.read_csv(pos_path)
    freq_dict = dict(zip(pos_df["word"], pos_df["count"]))
else:
    freq_dict = {}

if not freq_dict:
    from collections import Counter
    stop = {"the","and","is","in","it","of","a","to","i","this","have","for",
            "with","was","my","but","are","be","that","not","they","good","great"}
    c = Counter()
    for text in df[df["SentimentLabel"]=="Positive"]["combined"].sample(30000, random_state=1):
        c.update([w for w in re.sub(r"[^a-z ]", " ", text.lower()).split()
                  if w not in stop and len(w) > 2])
    freq_dict = dict(c.most_common(200))

wc = WordCloud(width=1000, height=500, background_color="white",
               colormap="Greens", max_words=150,
               prefer_horizontal=0.9).generate_from_frequencies(freq_dict)
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud – Positive Reviews", fontsize=15,
             fontweight="bold", color=DARK, pad=15)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig6_wordcloud_positive.png", dpi=150)
plt.close()
print("  Fig 6 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 7 – Helpfulness Ratio Distribution by Sentiment
# ══════════════════════════════════════════════════════════════════════════════
hr_df = df[df["HelpfulnessRatio"].notna() & (df["HelpfulnessDenominator"] >= 5)]
fig, ax = plt.subplots(figsize=(9, 5))
for lbl, clr in PALETTE.items():
    sub = hr_df[hr_df["SentimentLabel"] == lbl]["HelpfulnessRatio"]
    ax.hist(sub, bins=30, alpha=0.55, color=clr, label=lbl, density=True)
ax.set_xlabel("Helpfulness Ratio (helpful / total votes)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Helpfulness Ratio by Sentiment", fontsize=14,
             fontweight="bold", color=DARK)
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig7_helpfulness_by_sentiment.png", dpi=150)
plt.close()
print("  Fig 7 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 8 – Heatmap: Avg Score by Year × Sentiment Share
# ══════════════════════════════════════════════════════════════════════════════
pivot = (df[df["Year"].between(2002, 2012)]
         .groupby(["Year", "SentimentLabel"])
         .size()
         .unstack(fill_value=0))
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_pct[["Positive","Neutral","Negative"]].T,
            annot=True, fmt=".1f", cmap="RdYlGn",
            linewidths=0.5, ax=ax, cbar_kws={"label": "% of Reviews"})
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Sentiment", fontsize=12)
ax.set_title("Sentiment Share by Year (%)", fontsize=14,
             fontweight="bold", color=DARK)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig8_sentiment_heatmap.png", dpi=150)
plt.close()
print("  Fig 8 saved")

print(f"\nAll figures saved to {FIG_DIR}")

# ── Summary Stats (for slide deck) ───────────────────────────────────────────
summary = {
    "total_reviews"    : int(len(df)),
    "total_users"      : int(df["UserId"].nunique()) if "UserId" in df.columns else "N/A",
    "avg_score"        : round(float(df["Score"].mean()), 2),
    "pct_positive"     : round(float((df["SentimentLabel"]=="Positive").mean()*100), 1),
    "pct_neutral"      : round(float((df["SentimentLabel"]=="Neutral").mean()*100), 1),
    "pct_negative"     : round(float((df["SentimentLabel"]=="Negative").mean()*100), 1),
    "date_range"       : "Oct 1999 – Oct 2012",
}
with open("./summary_stats.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Summary stats:", summary)


Loading data from MongoDB …
  568,454 rows loaded
  Fig 1 saved
  Fig 2 saved
  Fig 3 saved
  Fig 4 saved
  Fig 5 saved
  Fig 6 saved
  Fig 7 saved
  Fig 8 saved

All figures saved to ./figures
Summary stats: {'total_reviews': 568454, 'total_users': 'N/A', 'avg_score': 4.18, 'pct_positive': 78.1, 'pct_neutral': 7.5, 'pct_negative': 14.4, 'date_range': 'Oct 1999 – Oct 2012'}
